**Feature Extraction using TF-IDF**

## What is this step?
Machine learning models cannot understand words,they only understand numbers.
Feature extraction converts our cleaned text reviews into numerical representations
that ML models can learn from.

## What is TF-IDF?
**TF-IDF** stands for **Term Frequency - Inverse Document Frequency**.

- **TF (Term Frequency):** How often a word appears in a single review
- **IDF (Inverse Document Frequency):** How rare a word is across ALL reviews

Words that appear frequently in one review but rarely across all reviews
get a HIGH score, these are the most meaningful/distinctive words.

Common words like "the", "and", "is" appear everywhere so they get a LOW score
and are essentially ignored.

## Example
In a positive review: *"yo product ekdam ramro cha"*
- "ramro" → appears in many positive reviews, rare in negative → HIGH TF-IDF score
- "yo" → appears in almost every review → LOW TF-IDF score

## Golden Rule: Fit on Training Data Only
We fit TF-IDF ONLY on training data, then use it to transform val and test.
Fitting on all three would give the model a sneak peek at unseen data, this is
called **data leakage** and makes results dishonest.

## Output of this step
- A numerical matrix for train, val, and test splits
- A saved TF-IDF vectorizer (to reuse consistently across all future steps)


In [5]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import os

In [6]:
# Load cleaned data
train_df = pd.read_csv('../data/processed/train_cleaned.csv')
val_df = pd.read_csv('../data/processed/val_cleaned.csv')
test_df = pd.read_csv('../data/processed/test_cleaned.csv')

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:", train_df.columns.tolist())
print("\nSample cleaned reviews:")
print(train_df['cleaned_text'].head(3))

Train shape: (70000, 23)
Val shape: (15000, 23)
Test shape: (15000, 23)

Train columns: ['review_id', 'product_id', 'product_name', 'product_category', 'brand', 'product_price_npr', 'seller_name', 'delivery_partner', 'reviewer_location', 'payment_method', 'review_date', 'review_text', 'rating', 'sentiment_label', 'verified_purchase', 'helpful_count', 'review_length_chars', 'review_length_tokens', 'english_token_ratio', 'nepali_token_ratio', 'language_dominance', 'has_emoji', 'cleaned_text']

Sample cleaned reviews:
0    best purchase year door step delivery raamro h...
1    wow ekdam ramro product battery backup ramro f...
2    imported jasto feel auncha super product fitti...
Name: cleaned_text, dtype: str


In [7]:
# Fit TF-IDF on TRAIN ONLY
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrams + bigrams → captures "not good", "ekdam ramro"
    min_df=2,             # ignore terms appearing in only 1 review (cuts spelling-variation noise)
    sublinear_tf=True     # dampens very frequent terms
)

X_train = vectorizer.fit_transform(train_df['cleaned_text'])
X_val   = vectorizer.transform(val_df['cleaned_text'])
X_test  = vectorizer.transform(test_df['cleaned_text'])

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

X_train: (70000, 18142)
X_val:   (15000, 18142)
X_test:  (15000, 18142)
Vocabulary size: 18142


In [9]:
y_train = train_df['sentiment_label']
y_val   = val_df['sentiment_label']
y_test  = test_df['sentiment_label']

os.makedirs('../models', exist_ok=True)
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
joblib.dump((X_train, y_train), '../data/processed/train_features.pkl')
joblib.dump((X_val, y_val),     '../data/processed/val_features.pkl')
joblib.dump((X_test, y_test),   '../data/processed/test_features.pkl')

print("Saved vectorizer + train/val/test features ")

Saved vectorizer + train/val/test features 
